<a href="https://colab.research.google.com/github/NourhanFarag/nourhan-flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NourhanFarag/nourhan-flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

### My provisional lane: Refresh / Content Opportunity Scoring

I have provisionally selected **Lane 2: Refresh / Content Opportunity Scoring**. My project will investigate how observable search-performance, content-age, freshness, CTR, and engagement signals can help prioritize content pages for human review.

The main problem is that a content team may manage many pages but have limited time to inspect and improve them. Reviewing every page equally would be inefficient. I therefore want to create a ranked review queue that helps an editor decide which pages should be examined first for possible refresh, expansion, protection, pruning, consolidation, or monitoring.

I selected this lane because it connects the available data to a clear decision and action. The starter dataset contains page-level measurements such as impressions, clicks, CTR, average position, sessions, engagement rate, scroll rate, content age, freshness, word count, and recent performance trends. These signals may help identify pages that have meaningful visibility but also show evidence of decline, staleness, weak click capture, or weak engagement.

CTR and engagement will be used as supporting signals within the main refresh-opportunity lane rather than as a separate second project. For example, a visible page with lower-than-expected CTR or weak engagement may receive a reason code that helps explain why it deserves review.

The proposed output is a decision-support ranking rather than an automatic editing instruction. A high-ranked page would be reviewed by a human who considers its editorial, search, and business context before deciding whether any change is appropriate. The project will begin with a transparent rule-based baseline, and machine learning will only be considered useful if it improves the ranking decision in a measurable and explainable way.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### Research question

Which content pages should be reviewed first for possible refresh, expansion, protection, pruning, consolidation, or monitoring, based on observable search-performance, freshness, CTR, and engagement signals?

### Unit of analysis

The unit of analysis is **one pseudonymized content page**. Each row in the starter dataset represents one page and contains measurements collected over a trailing 90-day period.

### Decision to improve

The project aims to improve the decision:

> Which pages should be placed at the top of the editorial review queue?

A content team may manage many pages but have limited time available for review. The system should help the team focus first on pages where attention may have the greatest practical value.

### Who acts on the result?

The result would be used by a content editor, SEO specialist, or content operations reviewer.

### Action taken

The reviewer would inspect the highest-ranked pages and decide whether each page should be:

- refreshed or updated;
- expanded with missing information;
- protected because it is already performing well;
- consolidated with related content;
- pruned when it has little measurable value;
- monitored without immediate changes;
- left unchanged after human review.

The recommendation would support the reviewer’s judgment rather than automatically changing the page.

### Expected output

The proposed output is a **ranked review queue**. Each page should eventually include:

- a priority score;
- a suggested review action;
- reason codes explaining the recommendation;
- a confidence level.

CTR and engagement would be supporting signals. For example, a visible page may receive a reason code for low click capture or weak engagement, but those signals would be interpreted together with position, impressions, age, and freshness.

### Cost of a wrong recommendation

A **false positive** occurs when the system ranks a page highly even though it does not need attention. This could waste editorial time, delay work on more valuable pages, or lead to unnecessary changes that disturb a page already performing adequately.

A **false negative** occurs when the system fails to rank a page that genuinely deserves review. This could allow a valuable page to continue losing visibility, clicks, or engagement while the opportunity for an earlier intervention is missed.

Because editorial capacity is limited, false positives near the top of the ranking are especially costly. The project should therefore prioritize producing a trustworthy top review queue rather than attempting to flag every possible page.

### Why data or machine learning may help

A simple rule-based score will be used as the baseline. For example, a basic rule could prioritize pages that have meaningful impressions, are outdated, and show signs of decline.

However, page priority may depend on interactions among several signals, including impressions, average position, CTR, content age, freshness, sessions, engagement, scroll activity, and content depth. A data-driven model may identify combinations that are difficult to express with one fixed rule.

Machine learning would only be considered useful if it improves the ranking over the transparent baseline while remaining explainable and safe for human review. The purpose is not simply to train a model; it is to improve the allocation of limited editorial attention.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
from pathlib import Path

import pandas as pd

# Load the CSV locally when the repo is available.
# In Colab, fall back to the public copy in your GitHub repository.
local_path = Path("data/raw/content_refresh_anonymized.csv")
github_url = (
    "https://raw.githubusercontent.com/"
    "NourhanFarag/nourhan-flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

data_source = local_path if local_path.exists() else github_url
df = pd.read_csv(data_source)

# Confirm that the columns needed for this quick analysis exist.
required_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "sessions_90d",
    "engagement_rate",
    "scroll_rate",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"The dataset is missing required columns: {sorted(missing_columns)}"
    )

# ---------------------------------------------------------
# 1. Dataset coverage
# ---------------------------------------------------------
page_count = df["content_id"].nunique()
client_count = df["client_id"].nunique()

# ---------------------------------------------------------
# 2. Pages currently observed as declining
# ---------------------------------------------------------
declining = df["trend_direction"].eq("down")
declining_count = int(declining.sum())
declining_pct = declining.mean() * 100

# ---------------------------------------------------------
# 3. Transparent Lane 2 review signals
# These are screening rules, not proof that a page needs editing.
# ---------------------------------------------------------

declining_with_demand = (
    df["trend_direction"].eq("down")
    & df["impressions_90d"].ge(100)
)

stale_visible = (
    df["days_since_last_update"].ge(180)
    & df["impressions_90d"].ge(500)
)

# avg_position = 0 means missing position data, so only use positions 1–20.
low_ctr_visible = (
    df["impressions_90d"].ge(500)
    & df["avg_position"].between(1, 20, inclusive="both")
    & df["ctr"].lt(0.5)
)

# Missing engagement measurements are not automatically treated as low.
low_engagement_visible = (
    df["sessions_90d"].ge(30)
    & (
        (
            df["engagement_rate"].notna()
            & df["engagement_rate"].lt(30)
        )
        |
        (
            df["scroll_rate"].notna()
            & df["scroll_rate"].lt(30)
        )
    )
)

has_review_signal = (
    declining_with_demand
    | stale_visible
    | low_ctr_visible
    | low_engagement_visible
)

review_signal_count = int(has_review_signal.sum())
review_signal_pct = has_review_signal.mean() * 100

# Display three headline findings.
print(
    f"1. Dataset coverage: {page_count:,} pages "
    f"across {client_count:,} pseudonymized clients."
)

print(
    f"2. Observed decline: {declining_count:,} pages "
    f"({declining_pct:.1f}% of pages) have trend_direction = 'down'."
)

print(
    f"3. Review opportunity: {review_signal_count:,} pages "
    f"({review_signal_pct:.1f}% of pages) meet at least one "
    f"transparent Lane 2 screening rule."
)

# Supporting breakdown for the third finding.
reason_breakdown = pd.DataFrame(
    {
        "Review signal": [
            "Declining with demand",
            "Stale visible page",
            "Low-CTR visible page",
            "Low-engagement visible page",
        ],
        "Page count": [
            int(declining_with_demand.sum()),
            int(stale_visible.sum()),
            int(low_ctr_visible.sum()),
            int(low_engagement_visible.sum()),
        ],
    }
)

reason_breakdown["Percent of dataset"] = (
    reason_breakdown["Page count"] / len(df) * 100
).round(1)

reason_breakdown

1. Dataset coverage: 30,000 pages across 32 pseudonymized clients.
2. Observed decline: 16,262 pages (54.2% of pages) have trend_direction = 'down'.
3. Review opportunity: 18,521 pages (61.7% of pages) meet at least one transparent Lane 2 screening rule.


,Review signal,Page count,Percent of dataset
0,Declining with demand,13152,43.8
1,Stale visible page,17,0.1
2,Low-CTR visible page,9745,32.5
3,Low-engagement visible page,7113,23.7


### What these numbers suggest

The starter dataset contains **30,000 content pages across 32 pseudonymized clients**, providing a substantial page-level sample for investigating content-review priorities.

A total of **16,262 pages, or 54.2% of the dataset, are currently classified as having a downward impression trend**. This suggests that decline-related review is not a rare edge case and that content teams may need a systematic way to decide which declining pages deserve attention first.

Using transparent screening rules based on demand, freshness, CTR, and engagement, **18,521 pages, or 61.7% of the dataset, meet at least one possible review signal**. The largest supporting groups include 13,152 pages declining with measurable demand, 9,745 visible pages with low CTR, and 7,113 pages with weak measured engagement.

These groups overlap, so their counts should not be added together. The large number of candidates also shows why a simple yes-or-no flag may not be enough: an editor could not review all 18,521 pages at once. A ranked system could help prioritize the pages with the strongest combination of visibility, decline, weak click capture, and engagement concerns.

These figures are observational screening results. They show that many pages may deserve human review, but they do not prove that refreshing a particular page will improve its performance.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What this work may support

This project may identify **observed patterns and directional signals** associated with content pages that deserve human review. It may show that certain combinations of visibility, declining impressions, content age, freshness, CTR, position, and engagement are useful for prioritizing pages.

The output may support statements such as:

- Pages with particular observed signals were ranked as stronger review candidates.
- Some measurable characteristics were associated with higher or lower review priority.
- A data-driven ranking performed better or worse than a transparent rule-based baseline on the chosen evaluation metric.
- The ranked queue may help editors allocate limited review time more efficiently.
- CTR and engagement may provide useful supporting context when interpreted alongside visibility, average position, and sufficient traffic volume.

The results should be described as **observed, measured, directional, and decision-support evidence**. A recommendation means that a page may deserve inspection; it does not mean that the page definitely requires editing.

### What this work cannot establish

This project cannot prove that refreshing, expanding, rewriting, or pruning a page will cause its search performance to improve. The dataset is observational and does not contain a randomized experiment or a valid untreated control group.

The work also cannot claim that:

- it discovered or predicted Google’s ranking algorithm;
- a signal is a confirmed Google ranking factor;
- low CTR proves that a title or description is poor;
- weak engagement proves that the content itself is low quality;
- a declining page will recover after an update;
- a high-ranked recommendation guarantees a successful outcome;
- correlations between variables demonstrate causation.

Other explanations may exist, including seasonality, changes in search demand, competition, measurement noise, related pages absorbing traffic, or changes to the search-results page.

### Intended interpretation

The final ranking should be treated as a **human-in-the-loop review tool**. Editors or SEO specialists should examine the page, consider business and editorial context, and decide whether to act, monitor, or leave it unchanged.

Any claims will be limited to the available pseudonymized dataset, the defined observation window, and the evaluation procedure used in this project. Findings should not automatically be generalized to every website, client, topic, or future time period.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`